# Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

# Data processing and math
import pandas as pd
import numpy as np

# Statistics
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display handling
import warnings


#################
# CONFIGURATION #
#################

# Suppress warnings
warnings.filterwarnings('ignore')

# Set global plotting style
sns.set_theme(style = "whitegrid")


#################
# VISUALIZATION #
#################
palette_main = {
    "Community": "#D55E00",
    "Family":    "#0072B2"
}

# Transform Data

In [ ]:
# Load data
df = pd.read_csv('blame_praise_self_pilot_4_data.csv')

# Define factors
factors_map = {
    'racism.12-pAction': ('Bad',  'Community'),
    '12.2':              ('Bad',  'Family'),
    'racism.24-pAction': ('Good', 'Community'),
    '24.2':              ('Good', 'Family')
}

# Reshape wide to long
long_data = []

for _, row in df.iterrows():
    p_id = row['ID']
    
    for col, (upbringing, environment) in factors_map.items():
        if col in df.columns and pd.notna(row[col]):
            val = pd.to_numeric(row[col], errors = 'coerce')
            val = val - 7
            
            long_data.append({
                'ID': p_id,
                'Upbringing': upbringing,
                'Environment': environment,
                'Blame': val,
                'Gender': row.get('Gender'),
                'Age': row.get('Age'),
                'Education': row.get('Education')
            })

df_long = pd.DataFrame(long_data)
print(f"Transformation complete ({len(df_long)} Observations).")

# Calculate Descriptive Statistics

In [ ]:
# Define group factors and dependent variables
group_factors = ['Upbringing', 'Environment']
dependent_var = 'Blame'

# Calculate descriptive statistics
descriptive_stats = df_long.groupby(group_factors)[dependent_var].agg(['mean', 'std', 'count']).round(3)

# Display results
display(descriptive_stats)

# Perform ANOVAs

In [ ]:
dv = 'Blame'
print(f"ANOVA ({dv})")

# Define formula
formula = f"{dv} ~ C(Upbringing) * C(Environment)"

# Drop lines with NaN
df_anova = df_long.dropna(subset = [dv])

# Fit model
model = ols(formula, data = df_anova).fit()

# Run ANOVA
aov_table = sm.stats.anova_lm(model, typ = 2)

# Calculate effect sizes
aov_table['partial_eta_sq'] = aov_table['sum_sq'] / (aov_table['sum_sq'] + aov_table.loc['Residual', 'sum_sq'])

# Display results
display(aov_table.round(3))

# Generate Bar Plot

In [ ]:
plt.figure(figsize=(8, 6))
    
# Create graph
g = sns.catplot(data = df_long,
                x = "Upbringing",
                y = "Blame",
                hue = "Environment",
                kind = "bar",
                palette = palette_main,
                capsize = .05,
                height = 5,
                aspect = 1.2
               )

# Set axis labels and titles
plt.title('Mean Blame by Upbringing and Environment', fontsize = 16)
plt.ylabel('Mean Blame Score')
plt.xlabel('Upbringing')

# Add horizontal zero lines and set y-axis limits
for ax in g.axes.flat:
    ax.set_ylim(-6, 6)
    ax.axhline(0, color = 'black', linewidth = 0.8, linestyle = '--')

# Export graph
filename = 'blame_analysis_pilot_4_bar_plot.png'
plt.savefig(filename, dpi = 300, bbox_inches = 'tight')
plt.show()
print(f"Graph saved as '{filename}'.")